# Model evaluation

Compare several classifiers on the exact train/test split exported by `data_cleaning.ipynb`. The target classes are 0, 1-2, 3-5, 6-15, 16-19, 20-25, and 26+ bikes. Run the final save cell in that notebook first to create `data/train.parquet` and `data/test.parquet`.

## Imports and configuration

In [8]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_DIR = Path("../data")
FEATURE_COLUMNS = [
    "idstation",
    "lat",
    "lon",
    "slots_total",
    "hour",
    "day_of_week",
    "is_weekend",
]
TARGET_COLUMN = "bike_bin"

## Load the saved train and test data

These files come directly from the `train_test_split` call in the cleaning notebook; this notebook does not recreate or change the split.

In [9]:
train_data = pd.read_parquet(DATA_DIR / "train.parquet")
test_data = pd.read_parquet(DATA_DIR / "test.parquet")
required_columns = [*FEATURE_COLUMNS, TARGET_COLUMN]

for split_name, split_data in (("train", train_data), ("test", test_data)):
    missing_columns = sorted(set(required_columns) - set(split_data.columns))
    if missing_columns:
        raise ValueError(f"{split_name} data is missing columns: {missing_columns}")
    if split_data.empty:
        raise ValueError(f"{split_name} data is empty.")
    if split_data[required_columns].isna().any().any():
        raise ValueError(f"{split_name} data contains missing model inputs or targets.")

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN]
X_test = test_data[FEATURE_COLUMNS]
y_test = test_data[TARGET_COLUMN]

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
display(
    pd.DataFrame(
        {
            "train": y_train.value_counts(normalize=True).sort_index(),
            "test": y_test.value_counts(normalize=True).sort_index(),
        }
    ).fillna(0).round(3)
)

Training rows: 10,345
Test rows: 2,587


,train,test
bike_bin,,
0,0.041,0.041
1,0.115,0.115
2,0.217,0.217
3,0.531,0.532
4,0.065,0.065
5,0.027,0.027
6,0.004,0.004


## Train and compare models

Logistic regression uses standardized features. `train_accuracy` and `test_accuracy` show accuracy on the fitted and held-out data. `test_error_rate` is `1 - test_accuracy`; `generalization_gap` is `train_accuracy - test_accuracy` (a larger positive gap can indicate overfitting). Macro F1 gives each class equal weight and is useful for this imbalanced target.

In [10]:
models = {
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
        ),
    ),
    "Decision tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42,
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
}

majority_class = y_train.mode().iloc[0]
predictions_by_model = {
    "Majority-class baseline": pd.Series(majority_class, index=y_test.index)
}
train_predictions_by_model = {
    "Majority-class baseline": pd.Series(majority_class, index=y_train.index)
}

for model_name, estimator in models.items():
    estimator.fit(X_train, y_train)
    train_predictions_by_model[model_name] = estimator.predict(X_train)
    predictions_by_model[model_name] = estimator.predict(X_test)

comparison = pd.DataFrame(
    {
        "train_accuracy": {
            name: accuracy_score(y_train, predictions)
            for name, predictions in train_predictions_by_model.items()
        },
        "test_accuracy": {
            name: accuracy_score(y_test, predictions)
            for name, predictions in predictions_by_model.items()
        },
        "test_error_rate": {
            name: 1 - accuracy_score(y_test, predictions)
            for name, predictions in predictions_by_model.items()
        },
        "generalization_gap": {
            name: accuracy_score(y_train, train_predictions_by_model[name])
            - accuracy_score(y_test, predictions)
            for name, predictions in predictions_by_model.items()
        },
        "macro_f1": {
            name: f1_score(y_test, predictions, average="macro", zero_division=0)
            for name, predictions in predictions_by_model.items()
        },
    }
).sort_values("test_accuracy", ascending=False)
display(comparison.round(3))

,train_accuracy,test_accuracy,test_error_rate,generalization_gap,macro_f1
Random forest,0.837,0.727,0.273,0.110,0.646
Decision tree,0.820,0.718,0.282,0.102,0.643
Majority-class baseline,0.531,0.532,0.468,-0.000,0.099
Logistic regression,0.237,0.227,0.773,0.010,0.156


## Inspect the random forest results

In [ ]:
forest_predictions = predictions_by_model["Random forest"]
class_labels = sorted(set(y_train.unique()) | set(y_test.unique()))

print("Random forest classification report:")
print(
    classification_report(
        y_test,
        forest_predictions,
        labels=class_labels,
        target_names=[str(label) for label in class_labels],
        zero_division=0,
    )
)

print("Random forest confusion matrix (rows: actual, columns: predicted):")
display(
    pd.DataFrame(
        confusion_matrix(y_test, forest_predictions, labels=class_labels),
        index=pd.Index(class_labels, name="actual"),
        columns=pd.Index(class_labels, name="predicted"),
    )
)
